In [2]:
import requests
import json
from typing import List, Dict

class OllamaChatbot:
    def __init__(self, model_name="qwen3:14b", base_url="http://localhost:11434"):
        """
        初始化聊天機器人
        
        Args:
            model_name: Ollama模型名稱 (預設: llama3)
            base_url: Ollama服務的基礎URL (預設: http://localhost:11434)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.conversation_history: List[Dict[str, str]] = []
        
    def send_message(self, message: str) -> str:
        """
        發送訊息給LLM並獲取回應
        
        Args:
            message: 用戶輸入的訊息
            
        Returns:
            LLM的回應
        """
        # 將用戶訊息添加到對話歷史
        self.conversation_history.append({"role": "user", "content": message})
        
        # 準備請求資料
        payload = {
            "model": self.model_name,
            "messages": self.conversation_history,
            "stream": False
        }
        
        try:
            # 發送請求到Ollama
            response = requests.post(
                self.chat_url,
                json=payload,
                headers={"Content-Type": "application/json"},
                timeout=30000
            )
            
            if response.status_code == 200:
                result = response.json()
                assistant_message = result["message"]["content"]
                
                # 將助手回應添加到對話歷史
                self.conversation_history.append({
                    "role": "assistant", 
                    "content": assistant_message
                })
                
                return assistant_message
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except requests.exceptions.RequestException as e:
            return f"連接錯誤: {str(e)}"
        except json.JSONDecodeError as e:
            return f"JSON解析錯誤: {str(e)}"
        except Exception as e:
            return f"未知錯誤: {str(e)}"
    
    def new_session(self):
        """
        重置對話記憶，開始新的對話會話
        """
        self.conversation_history = []
        print("✅ 對話記憶已重置，開始新的會話！")
    
    def show_history(self):
        """
        顯示當前對話歷史
        """
        if not self.conversation_history:
            print("📝 目前沒有對話記錄")
            return
            
        print("\n📚 對話歷史:")
        print("-" * 50)
        for i, msg in enumerate(self.conversation_history, 1):
            role = "用戶" if msg["role"] == "user" else "助手"
            print(f"{i}. {role}: {msg['content']}")
        print("-" * 50)
    
    def check_model(self) -> bool:
        """
        檢查模型是否可用
        
        Returns:
            模型是否可用
        """
        try:
            models_url = f"{self.base_url}/api/tags"
            response = requests.get(models_url, timeout=10)
            
            if response.status_code == 200:
                models = response.json().get("models", [])
                available_models = [model["name"] for model in models]
                
                if self.model_name in available_models:
                    print(f"✅ 模型 {self.model_name} 已就緒")
                    return True
                else:
                    print(f"❌ 模型 {self.model_name} 不可用")
                    print(f"可用模型: {available_models}")
                    return False
            else:
                print(f"❌ 無法連接到Ollama服務: HTTP {response.status_code}")
                return False
                
        except Exception as e:
            print(f"❌ 檢查模型時發生錯誤: {str(e)}")
            return False

def main():
    """
    主要執行函數
    """
    print("🤖 Ollama Llama3 聊天機器人")
    print("=" * 50)
    
    # 初始化聊天機器人
    chatbot = OllamaChatbot()
    
    # 檢查模型可用性
    if not chatbot.check_model():
        print("\n請確保:")
        print("1. Ollama服務正在運行 (ollama serve)")
        print("2. 已下載llama3模型 (ollama pull llama3)")
        return
    
    print("\n使用說明:")
    print("• 直接輸入訊息與AI對話")
    print("• 輸入 'New Session' 重置對話記憶")
    print("• 輸入 'history' 查看對話歷史")
    print("• 輸入 'quit' 或 'exit' 退出程式")
    print("-" * 50)
    
    # 主對話迴圈
    while True:
        try:
            user_input = input("\n👤 您: ").strip()
            
            # 退出指令
            if user_input.lower() in ['quit', 'exit', '退出', 'q']:
                print("👋 再見！")
                break
            
            # 空輸入處理
            if not user_input:
                print("⚠️  請輸入訊息")
                continue
            
            # 新會話指令
            if user_input.lower() == 'new session':
                chatbot.new_session()
                continue
            
            # 查看歷史指令
            if user_input.lower() == 'history':
                chatbot.show_history()
                continue
            
            # 發送訊息並獲取回應
            print("🤔 思考中...")
            response = chatbot.send_message(user_input)
            print(f"🤖 助手: {response}")
            
        except KeyboardInterrupt:
            print("\n\n👋 程式已中斷，再見！")
            break
        except Exception as e:
            print(f"❌ 發生錯誤: {str(e)}")

if __name__ == "__main__":
    main()

🤖 Ollama Llama3 聊天機器人
✅ 模型 qwen3:14b 已就緒

使用說明:
• 直接輸入訊息與AI對話
• 輸入 'New Session' 重置對話記憶
• 輸入 'history' 查看對話歷史
• 輸入 'quit' 或 'exit' 退出程式
--------------------------------------------------



👤 您:  我目前在世新大學機械系的大三就讀。


🤔 思考中...
🤖 助手: 你好！恭喜你正在世新大學機械系就讀，大三是個非常重要的階段，許多同學會開始思考未來的職業方向、實習機會、課業規劃，甚至是否要準備研究所考試或出國深造。如果你有任何問題，例如：

- **課業相關**（例如：機械設計、製造工程、材料科學等課程的學習困難）
- **實習與就業**（例如：如何找相關實習、產業趨勢、履歷撰寫）
- **未來規劃**（例如：是否要考研究所、出國、轉職其他領域）
- **課外活動**（例如：如何參與競賽、專題計畫、學生會等）

都可以告訴我，我會盡力協助你！ 😊  
如果你目前還沒想好要問什麼，也可以先分享一下你最近在忙什麼，或者對機械工程有什麼興趣的領域，我們可以一起討論～



👤 您:  請將我的學校與系所英文名稱轉換為 TOON 格式回傳。  TOON 格式範例： users[2]{id,name,role}:   1,Alice,admin   2,Bob,user  規則： 僅回傳 TOON 格式內容。 嚴禁任何開場白、解釋或 Markdown 代碼塊標記。 直接輸出結果。


🤔 思考中...
🤖 助手: users[2]{id,name,role}:  
1,Shih Hsin University,school  
2,Department of Mechanical Engineering,department



👤 您:  quit


👋 再見！


In [ ]:
import requests
import json
from typing import List, Dict

class OllamaChatbot:
    def __init__(self, model_name="qwen3:14b", base_url="http://localhost:11434"):
        """
        初始化聊天機器人
        
        Args:
            model_name: Ollama模型名稱 (預設: llama3)
            base_url: Ollama服務的基礎URL (預設: http://localhost:11434)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.conversation_history: List[Dict[str, str]] = []
        
    def send_message(self, message: str) -> str:
        """
        發送訊息給LLM並獲取回應
        
        Args:
            message: 用戶輸入的訊息
            
        Returns:
            LLM的回應
        """
        # 將用戶訊息添加到對話歷史
        self.conversation_history.append({"role": "user", "content": message})
        
        # 準備請求資料
        payload = {
            "model": self.model_name,
            "messages": self.conversation_history,
            "stream": False
        }
        
        try:
            # 發送請求到Ollama
            response = requests.post(
                self.chat_url,
                json=payload,
                headers={"Content-Type": "application/json"},
                timeout=30000
            )
            
            if response.status_code == 200:
                result = response.json()
                assistant_message = result["message"]["content"]
                
                # 將助手回應添加到對話歷史
                self.conversation_history.append({
                    "role": "assistant", 
                    "content": assistant_message
                })
                
                return assistant_message
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except requests.exceptions.RequestException as e:
            return f"連接錯誤: {str(e)}"
        except json.JSONDecodeError as e:
            return f"JSON解析錯誤: {str(e)}"
        except Exception as e:
            return f"未知錯誤: {str(e)}"
    
    def new_session(self):
        """
        重置對話記憶，開始新的對話會話
        """
        self.conversation_history = []
        print("✅ 對話記憶已重置，開始新的會話！")
    
    def show_history(self):
        """
        顯示當前對話歷史
        """
        if not self.conversation_history:
            print("📝 目前沒有對話記錄")
            return
            
        print("\n📚 對話歷史:")
        print("-" * 50)
        for i, msg in enumerate(self.conversation_history, 1):
            role = "用戶" if msg["role"] == "user" else "助手"
            print(f"{i}. {role}: {msg['content']}")
        print("-" * 50)
    
    def check_model(self) -> bool:
        """
        檢查模型是否可用
        
        Returns:
            模型是否可用
        """
        try:
            models_url = f"{self.base_url}/api/tags"
            response = requests.get(models_url, timeout=10)
            
            if response.status_code == 200:
                models = response.json().get("models", [])
                available_models = [model["name"] for model in models]
                
                if self.model_name in available_models:
                    print(f"✅ 模型 {self.model_name} 已就緒")
                    return True
                else:
                    print(f"❌ 模型 {self.model_name} 不可用")
                    print(f"可用模型: {available_models}")
                    return False
            else:
                print(f"❌ 無法連接到Ollama服務: HTTP {response.status_code}")
                return False
                
        except Exception as e:
            print(f"❌ 檢查模型時發生錯誤: {str(e)}")
            return False

def main(model_name="qwen3:14b"):
    """
    主要執行函model_name數
    """
    print("🤖 Ollama {model_name} 聊天機器人")
    print("=" * 50)
    
    # 初始化聊天機器人
    chatbot = OllamaChatbot(model_name)
    
    # 檢查模型可用性
    if not chatbot.check_model():
        print("\n請確保:")
        print("1. Ollama服務正在運行 (ollama serve)")
        print("2. 已下載llama3模型 (ollama pull llama3)")
        return
    
    print("\n使用說明:")
    print("• 直接輸入訊息與AI對話")
    print("• 輸入 'New Session' 重置對話記憶")
    print("• 輸入 'history' 查看對話歷史")
    print("• 輸入 'quit' 或 'exit' 退出程式")
    print("-" * 50)
    
    # 主對話迴圈
    while True:
        try:
            user_input = input("\n👤 您: ").strip()
            
            # 退出指令
            if user_input.lower() in ['quit', 'exit', '退出', 'q']:
                print("👋 再見！")
                break
            
            # 空輸入處理
            if not user_input:
                print("⚠️  請輸入訊息")
                continue
            
            # 新會話指令
            if user_input.lower() == 'new session':
                chatbot.new_session()
                continue
            
            # 查看歷史指令
            if user_input.lower() == 'history':
                chatbot.show_history()
                continue
            
            # 發送訊息並獲取回應
            print("🤔 思考中...")
            response = chatbot.send_message(user_input)
            print(f"🤖 助手: {response}")
            
        except KeyboardInterrupt:
            print("\n\n👋 程式已中斷，再見！")
            break
        except Exception as e:
            print(f"❌ 發生錯誤: {str(e)}")

if __name__ == "__main__":
    main(model_name="qwen-cpu-3-30b:latest")

🤖 Ollama {model_name} 聊天機器人
✅ 模型 qwen-cpu-3-30b:latest 已就緒

使用說明:
• 直接輸入訊息與AI對話
• 輸入 'New Session' 重置對話記憶
• 輸入 'history' 查看對話歷史
• 輸入 'quit' 或 'exit' 退出程式
--------------------------------------------------



👤 您:  我目前在世新大學機械系的大三就讀。


🤔 思考中...
🤖 助手: 你好！很高興認識你是世新大學機械系的大三學生。

在機械系學習期間，你可能會接觸到很多有趣的課程，比如：

- 機械設計與製造
- 熱力學與流體力學
- 材料科學
- 自動控制理論
- CAD/CAM 軟體應用

你目前對哪個領域比較感興趣呢？是偏向傳統機械設計、自動化控制，還是對新興的AI結合機械領域有興趣？我很好奇你在學業上有哪些特別的學習經驗或挑戰。

另外，作為大三學生，你是否已經開始思考未來的發展方向了？是想要繼續深造、進入產業界，還是有其他規劃呢？



👤 您:  請將我的學校與系所英文名稱轉換為 TOON 格式回傳。  TOON 格式範例： users[2]{id,name,role}:   1,Alice,admin   2,Bob,user  規則： 僅回傳 TOON 格式內容。 嚴禁任何開場白、解釋或 Markdown 代碼塊標記。 直接輸出結果。


🤔 思考中...
🤖 助手: users[1]{id,name,role}: 1,Shih Shin University Mechanical Engineering Department,student



👤 您:  請將我的學校與系所英文名稱轉換為 TOON 格式回傳。  TOON 格式範例： users[2]{id,attribute,value}:   1,school,National Taiwan University   2,department,Department of Finance  規則： 僅回傳 TOON 格式內容。 嚴禁任何開場白、解釋或 Markdown 代碼塊標記。 直接輸出結果。


🤔 思考中...
🤖 助手: users[2]{id,attribute,value}: 1,school,Shih Shin University 2,department,Department of Mechanical Engineering
